# siebel_id_update

Incremental update logic for `shebang.output` only.

Scope:
- helper functions for graph expansion and table swaps
- fallback mutation pattern for environments where SQL UPDATE is unsupported on Parquet tables
- one public entrypoint: `run_incremental_update(...)`

## Memory-safety notes

Use these knobs when data grows:
- Keep `checkpoint_every` small (for long traversals) to break lineage.
- Execute updates in source-side batches when ingesting many mutations.
- Avoid broad `.collect()`; keep validation samples bounded with `limit(...)`.
- Refresh tables and clear cache after each run in iterative pipelines.

## Incremental Flow Diagram

This module updates only impacted rows instead of recomputing everything.

Flow summary:
1. Read current source snapshots from `source_db.direct_bank` and `source_db.ggm_np`.
2. Build current graph edges (`rel_id <-> id_key`) and active seed set.
3. Load previous state snapshots from `state_db` (`edge_snapshot`, `seed_snapshot`, `seed_membership`).
4. Detect deltas:
   - added/removed edges
   - added/removed seeds
5. Compute impacted component:
   - start from changed rel_ids + changed seeds + rows previously mapped to removed seeds
   - expand through the graph to get all reachable impacted rel_ids
6. Recompute only impacted mappings:
   - expand `(seed_rel_id, rel_id)` reachability
   - assign deterministic `REL_ID_REGIE_KLANT` per rel_id (numeric min when possible, else lexical min)
7. Merge output:
   - keep old unimpacted rows
   - replace impacted rows with recomputed rows
   - keep only rows whose `REL_ID_REGIE_KLANT` is still an active seed
8. Persist results with staging-table swaps:
   - `target_db.output`
   - `state_db.edge_snapshot`
   - `state_db.seed_snapshot`
   - `state_db.seed_membership`
9. Return run stats for observability (`changed_edges`, `added_seeds`, `removed_seeds`, `impacted_rel_ids`, `output_rows`).

Pseudo-flow:

```text
source tables -> current edges/seeds ----+
                                      compare with previous state
                                                |
                                                v
                                     impacted start set
                                                |
                                       graph expansion
                                                |
                                    impacted rel_id set
                                                |
                                   recompute impacted output
                                                |
                            old unimpacted + recomputed impacted
                                                |
                                       write output + state
```

In [ ]:
from pathlib import Path
from pyspark.sql import DataFrame, SparkSession, functions as F


def _has_rows(df: DataFrame) -> bool:
    """Cheap emptiness check: trigger only a 1-row job instead of a full count."""
    return df.limit(1).count() > 0


def overwrite_table_with_staging_swap(spark: SparkSession, full_table_name: str, df: DataFrame, fmt: str = 'parquet') -> None:
    """
    Replace a table using a staging table and rename swap.

    Why this pattern:
    - Local Hive/Parquet setups often do not support UPDATE/MERGE robustly.
    - Renaming a fully written staging table is safer than mutating in place.
    """
    stage = f"{full_table_name}__staging"

    # Always start from a clean staging table name.
    spark.sql(f"DROP TABLE IF EXISTS {stage}")

    # Write full replacement snapshot.
    df.write.mode('overwrite').format(fmt).saveAsTable(stage)

    # Swap: drop old table name, then move staging into final name.
    spark.sql(f"DROP TABLE IF EXISTS {full_table_name}")
    spark.sql(f"ALTER TABLE {stage} RENAME TO {full_table_name}")

    # Refresh catalog metadata so subsequent reads see latest files.
    spark.catalog.refreshTable(full_table_name)


def mutate_table_with_overwrite_swap(spark: SparkSession, full_table_name: str, transform_fn) -> None:
    """
    Fallback mutation primitive:
    1) read table
    2) apply DataFrame transform
    3) replace table via staging swap
    """
    current_df = spark.table(full_table_name)
    next_df = transform_fn(current_df)
    overwrite_table_with_staging_swap(spark, full_table_name, next_df)


def _checkpoint(df: DataFrame, enabled: bool) -> DataFrame:
    """
    Materialize lineage when enabled.

    Checkpointing breaks long dependency chains in iterative graph traversal,
    reducing memory pressure and stale-plan issues in local Spark.
    """
    return df.checkpoint(eager=True) if enabled else df

In [ ]:
def expand_rel_component(start_rel_df: DataFrame, edges_df: DataFrame, max_iter: int = 40, checkpoint_every: int = 2) -> DataFrame:
    """
    Expand impacted REL_ID component with undirected traversal over rel_id<->id_key edges.

    Input:
    - start_rel_df: changed/impacted REL_ID seeds
    - edges_df: deduplicated rel_id/id_key edge list

    Output:
    - all reachable REL_ID values from the impacted start set
    """
    # Frontier starts with initial impacted rel_ids.
    frontier = start_rel_df.select('rel_id').dropDuplicates(['rel_id'])
    frontier = _checkpoint(frontier, checkpoint_every > 0)

    # Visited accumulates all discovered rel_ids.
    visited = frontier

    for step in range(max_iter):
        # Step A: rel_id -> id_key
        via_id = frontier.join(edges_df, 'rel_id', 'inner').select('id_key').dropDuplicates(['id_key'])

        # Step B: id_key -> rel_id
        next_rel = via_id.join(edges_df, 'id_key', 'inner').select('rel_id').dropDuplicates(['rel_id'])

        # Keep only newly discovered rel_ids.
        new_rel = next_rel.join(visited, 'rel_id', 'left_anti')
        if not _has_rows(new_rel):
            break

        # Grow visited/component and continue from new frontier.
        visited = visited.unionByName(new_rel).dropDuplicates(['rel_id'])
        frontier = new_rel

        # Periodically materialize to avoid long lineage.
        if checkpoint_every > 0 and (step + 1) % checkpoint_every == 0:
            visited = _checkpoint(visited, True)
            frontier = _checkpoint(frontier, True)

    return visited


def expand_seed_pairs(active_seeds_df: DataFrame, edges_df: DataFrame, max_iter: int = 40, checkpoint_every: int = 2) -> DataFrame:
    """
    Expand (seed_rel_id, rel_id) reachability pairs for active seeds.

    This keeps seed identity during traversal so we can later assign each rel_id
    to the deterministic REGIE seed.
    """
    # Each seed initially reaches itself.
    frontier = active_seeds_df.select(F.col('seed_rel_id'), F.col('seed_rel_id').alias('rel_id'))
    frontier = _checkpoint(frontier, checkpoint_every > 0)
    visited = frontier

    for step in range(max_iter):
        # Step A: (seed, rel_id) -> (seed, id_key)
        via_id = (
            frontier.alias('f')
            .join(edges_df.alias('e'), F.col('f.rel_id') == F.col('e.rel_id'), 'inner')
            .select(F.col('f.seed_rel_id'), F.col('e.id_key'))
            .dropDuplicates(['seed_rel_id', 'id_key'])
        )

        # Step B: (seed, id_key) -> (seed, rel_id)
        next_rel = (
            via_id.alias('i')
            .join(edges_df.alias('e'), 'id_key', 'inner')
            .select(F.col('i.seed_rel_id'), F.col('e.rel_id'))
            .dropDuplicates(['seed_rel_id', 'rel_id'])
        )

        # Keep only new seed/rel pairs.
        new_pairs = next_rel.join(visited, ['seed_rel_id', 'rel_id'], 'left_anti')
        if not _has_rows(new_pairs):
            break

        visited = visited.unionByName(new_pairs).dropDuplicates(['seed_rel_id', 'rel_id'])
        frontier = new_pairs

        if checkpoint_every > 0 and (step + 1) % checkpoint_every == 0:
            visited = _checkpoint(visited, True)
            frontier = _checkpoint(frontier, True)

    return visited

In [ ]:
def _ensure_tables(spark: SparkSession, target_db: str, state_db: str) -> None:
    """
    Ensure state/output databases and tables exist.

    state_db tables persist previous run snapshots used for delta detection:
    - edge_snapshot
    - seed_snapshot
    - seed_membership
    """
    try:
        spark.sql(f"CREATE DATABASE IF NOT EXISTS {state_db}")
    except Exception:
        # Some restricted catalogs may reject DB create; table creation below can still work.
        pass

    try:
        spark.sql(f"CREATE DATABASE IF NOT EXISTS {target_db}")
    except Exception:
        pass

    spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {state_db}.edge_snapshot (
      source_table STRING,
      rel_id STRING,
      id_key STRING
    ) USING PARQUET
    """)

    spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {state_db}.seed_snapshot (
      seed_rel_id STRING
    ) USING PARQUET
    """)

    spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {state_db}.seed_membership (
      rel_id STRING,
      rel_id_regie_klant STRING
    ) USING PARQUET
    """)

    spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {target_db}.output (
      REL_ID STRING,
      REL_ID_REGIE_KLANT STRING
    ) USING PARQUET
    """)

In [ ]:
def run_incremental_update(
    spark: SparkSession,
    source_db: str = 'shebang',
    target_db: str = 'shebang',
    state_db: str = 'shebang_state',
    max_iter: int = 40,
    checkpoint_every: int = 2,
    show_samples: bool = False,
    checkpoint_dir: str | None = None,
) -> dict:
    """
    Incrementally refresh REL_ID -> REL_ID_REGIE_KLANT mapping.

    High-level flow:
    1) Build current edge/seed snapshots from source tables.
    2) Compare against previous state snapshots to detect deltas.
    3) Compute impacted graph component only (avoid full reprocessing).
    4) Recompute mapping only for impacted REL_ID values.
    5) Merge with untouched output rows and persist via staging swaps.
    6) Save current snapshots as next-run state.
    """
    # Optional lineage control for iterative traversal.
    if checkpoint_every > 0:
        ckpt_dir = checkpoint_dir or str((Path.cwd() / '.local' / 'spark' / 'checkpoints' / 'notebook_update').resolve())
        Path(ckpt_dir).mkdir(parents=True, exist_ok=True)
        try:
            spark.sparkContext.setCheckpointDir(ckpt_dir)
        except Exception:
            # Some local/security-managed Spark contexts may reject checkpoint configuration.
            checkpoint_every = 0

    _ensure_tables(spark, target_db, state_db)

    # -----------------------------
    # 1) Build current source snapshot
    # -----------------------------
    direct = spark.table(f"{source_db}.direct_bank")
    ggm = spark.table(f"{source_db}.ggm_np")

    # Unified edge list from both sources:
    # - direct_bank: rel_id <-> np_sbl_id
    # - ggm_np:      rel_id <-> ikb_no
    edges_cur = (
        direct.select(F.lit('direct_bank').alias('source_table'), F.col('rel_id').cast('string').alias('rel_id'), F.col('np_sbl_id').cast('string').alias('id_key'))
        .unionByName(ggm.select(F.lit('ggm_np').alias('source_table'), F.col('rel_id').cast('string').alias('rel_id'), F.col('ikb_no').cast('string').alias('id_key')))
        .filter(F.col('rel_id').isNotNull() & F.col('id_key').isNotNull() & (F.trim(F.col('rel_id')) != '') & (F.trim(F.col('id_key')) != ''))
        .dropDuplicates(['source_table', 'rel_id', 'id_key'])
    )

    # Active REGIE seeds from direct_bank business rule.
    seeds_cur = (
        direct
        .filter((F.col('drc_bnk_f') == 'Y') & (F.col('edl_valid_to_dts').cast('string').startswith('9999-12-31')))
        .select(F.col('rel_id').cast('string').alias('seed_rel_id'))
        .dropDuplicates(['seed_rel_id'])
    )

    # Previous run state snapshots.
    edges_prev = spark.table(f"{state_db}.edge_snapshot").dropDuplicates(['source_table', 'rel_id', 'id_key'])
    seeds_prev = spark.table(f"{state_db}.seed_snapshot").dropDuplicates(['seed_rel_id'])
    membership_prev = spark.table(f"{state_db}.seed_membership").dropDuplicates(['rel_id', 'rel_id_regie_klant'])

    # -----------------------------
    # 2) Detect deltas (added/removed edges/seeds)
    # -----------------------------
    edge_cols = ['source_table', 'rel_id', 'id_key']
    edge_added = edges_cur.select(*edge_cols).exceptAll(edges_prev.select(*edge_cols))
    edge_removed = edges_prev.select(*edge_cols).exceptAll(edges_cur.select(*edge_cols))
    edge_changed = edge_added.unionByName(edge_removed).dropDuplicates(edge_cols)

    seed_added = seeds_cur.select('seed_rel_id').exceptAll(seeds_prev.select('seed_rel_id'))
    seed_removed = seeds_prev.select('seed_rel_id').exceptAll(seeds_cur.select('seed_rel_id'))

    # Any rel_id previously mapped to removed seeds must be reconsidered.
    affected_rel_from_removed = (
        membership_prev
        .join(seed_removed, membership_prev.rel_id_regie_klant == seed_removed.seed_rel_id, 'inner')
        .select('rel_id')
        .dropDuplicates(['rel_id'])
    )

    # Starting points for impact propagation.
    impact_start_rel = (
        edge_changed.select('rel_id')
        .unionByName(seed_added.select(F.col('seed_rel_id').alias('rel_id')))
        .unionByName(seed_removed.select(F.col('seed_rel_id').alias('rel_id')))
        .unionByName(affected_rel_from_removed.select('rel_id'))
        .dropDuplicates(['rel_id'])
    )

    edges_full = edges_cur.select('rel_id', 'id_key').dropDuplicates(['rel_id', 'id_key'])

    # -----------------------------
    # 3) Compute impacted REL_ID component
    # -----------------------------
    if not _has_rows(edges_prev) and not _has_rows(seeds_prev) and not _has_rows(membership_prev):
        # First run: full compute.
        impacted_rel_cur = edges_full.select('rel_id').dropDuplicates(['rel_id'])
    elif not _has_rows(impact_start_rel):
        # No deltas: no impacted rel_ids.
        impacted_rel_cur = spark.createDataFrame([], 'rel_id string')
    else:
        # Expand changed component only.
        impacted_rel_cur = expand_rel_component(impact_start_rel, edges_full, max_iter=max_iter, checkpoint_every=checkpoint_every)

    # Seeds previously linked to newly impacted rel_ids.
    impacted_seeds_prev = (
        membership_prev
        .join(impacted_rel_cur, 'rel_id', 'inner')
        .select(F.col('rel_id_regie_klant').alias('seed_rel_id'))
        .dropDuplicates(['seed_rel_id'])
    )

    # Final impacted seed set = historical impacted + newly added/removed seeds.
    impacted_seeds = (
        impacted_seeds_prev
        .unionByName(seed_added.select('seed_rel_id'))
        .unionByName(seed_removed.select('seed_rel_id'))
        .dropDuplicates(['seed_rel_id'])
    )

    # Only active seeds are allowed to contribute in current run.
    impacted_active_seeds = impacted_seeds.join(seeds_cur, 'seed_rel_id', 'inner')

    # Include rows previously mapped by impacted seeds, so we can replace/remove them safely.
    impacted_rel_prev = (
        membership_prev
        .join(impacted_seeds.withColumnRenamed('seed_rel_id', 'rel_id_regie_klant'), 'rel_id_regie_klant', 'inner')
        .select('rel_id')
        .dropDuplicates(['rel_id'])
    )

    # Union current and historical impacted rel_ids.
    impacted_rel_all = impacted_rel_cur.unionByName(impacted_rel_prev).dropDuplicates(['rel_id'])

    # -----------------------------
    # 4) Recompute mapping only for impacted area
    # -----------------------------
    if not _has_rows(impacted_active_seeds):
        # No active seeds contributing -> recomputed set is empty.
        recomputed = spark.createDataFrame([], 'REL_ID string, REL_ID_REGIE_KLANT string')
    else:
        # Expand (seed, rel_id) reachability inside full edge graph.
        seed_pairs = expand_seed_pairs(impacted_active_seeds, edges_full, max_iter=max_iter, checkpoint_every=checkpoint_every)

        # Keep only impacted rel_ids we intend to replace.
        seed_pairs = seed_pairs.join(impacted_rel_all, 'rel_id', 'inner')

        # Deterministic assignment when rel_id reachable from multiple seeds:
        # prefer numeric minimum when possible, otherwise lexical minimum.
        recomputed = (
            seed_pairs
            .withColumn('seed_num', F.col('seed_rel_id').cast('bigint'))
            .groupBy('rel_id')
            .agg(F.min(F.when(F.col('seed_num').isNotNull(), F.col('seed_num'))).alias('min_seed_num'), F.min('seed_rel_id').alias('min_seed_lex'))
            .withColumn('REL_ID_REGIE_KLANT', F.when(F.col('min_seed_num').isNotNull(), F.col('min_seed_num').cast('string')).otherwise(F.col('min_seed_lex')))
            .select(F.col('rel_id').alias('REL_ID'), 'REL_ID_REGIE_KLANT')
            .dropDuplicates(['REL_ID'])
        )

    # Materialize metrics before swaps to avoid stale-file lineage reads.
    changed_edges_count = edge_changed.count()
    added_seeds_count = seed_added.count()
    removed_seeds_count = seed_removed.count()
    impacted_rel_count = impacted_rel_all.count()

    # -----------------------------
    # 5) Merge recomputed impacted rows with untouched output
    # -----------------------------
    output_prev = spark.table(f"{target_db}.output").select(F.col('REL_ID').cast('string').alias('REL_ID'), F.col('REL_ID_REGIE_KLANT').cast('string').alias('REL_ID_REGIE_KLANT'))
    if not _has_rows(impacted_rel_all):
        output_new = output_prev
    else:
        # Keep only rows not impacted, then append recomputed impacted rows.
        old_unimpacted = output_prev.join(impacted_rel_all.select(F.col('rel_id').alias('REL_ID')), 'REL_ID', 'left_anti')
        output_new = old_unimpacted.unionByName(recomputed).dropDuplicates(['REL_ID'])

    # Enforce that target seeds are still active in current snapshot.
    output_new = output_new.join(seeds_cur.select(F.col('seed_rel_id').alias('REL_ID_REGIE_KLANT')), 'REL_ID_REGIE_KLANT', 'inner')

    # Persist final output snapshot.
    overwrite_table_with_staging_swap(spark, f"{target_db}.output", output_new)
    output_new_local = spark.table(f"{target_db}.output").select(F.col('REL_ID').cast('string').alias('REL_ID'), F.col('REL_ID_REGIE_KLANT').cast('string').alias('REL_ID_REGIE_KLANT'))

    # -----------------------------
    # 6) Persist next-run state snapshots
    # -----------------------------
    overwrite_table_with_staging_swap(spark, f"{state_db}.edge_snapshot", edges_cur)
    overwrite_table_with_staging_swap(spark, f"{state_db}.seed_snapshot", seeds_cur)
    overwrite_table_with_staging_swap(
        spark,
        f"{state_db}.seed_membership",
        output_new_local.select(F.col('REL_ID').alias('rel_id'), F.col('REL_ID_REGIE_KLANT').alias('rel_id_regie_klant')),
    )

    # Drop cached plans between iterative notebook runs.
    spark.catalog.clearCache()

    stats = {
        'changed_edges': changed_edges_count,
        'added_seeds': added_seeds_count,
        'removed_seeds': removed_seeds_count,
        'impacted_rel_ids': impacted_rel_count,
        'output_rows': output_new_local.count(),
    }

    if show_samples:
        print('Run stats:', stats)
        display(output_new_local.orderBy('REL_ID').limit(50).toPandas())

    return stats